# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a demonstration of loading and exploring the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section lists all available record sets in the dataset and prints sample records from each. All entities are referenced by their `@id` fields.

In [ ]:
# List all available record sets by @id
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
print("Available Record Sets (@id):")
for i, rs_id in enumerate(record_sets):
    print(f"[{i}] {rs_id}")

# Print a sample record from each record set (if any exist)
for rs_id in record_sets:
    print(f"\nSample records for RecordSet: {rs_id}")
    records_iter = dataset.records(record_set=rs_id)
    for i, rec in enumerate(records_iter):
        print(rec)
        if i >= 2:
            break
    if i == 0:
        print("(No records found in this RecordSet)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` values for both the record set and its fields.

- We extract all record sets into a dictionary of DataFrames, keyed by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nColumns for Record Set {rs_id}:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"\n(No records found for Record Set {rs_id})")

# For analysis, let's pick the main tabular record set (the one with most columns and rows)
main_record_set_id = None
for rs_id, df in dataframes.items():
    if main_record_set_id is None or df.shape[1] > dataframes[main_record_set_id].shape[1]:
        main_record_set_id = rs_id

print(f"\nChosen main RecordSet for analysis: {main_record_set_id}")
main_df = dataframes[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll identify some numeric columns and categorical columns programmatically, then demonstrate filtering, normalization, and grouping.

All fields are referenced by their `@id` (i.e., column names in the DataFrame).

In [ ]:
# Identify likely numeric and categorical columns
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    # Try to convert all columns to float where possible
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
        except (ValueError, TypeError):
            continue
    numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()

print("Numeric field candidates (@id):", numeric_candidates)

# Choose a numeric field to analyze
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    raise ValueError("No numeric field found in the main record set.")

threshold = main_df[numeric_field_id].quantile(0.5)  # median split
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Identify a group (categorical) field
categorical_candidates = main_df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for cand in categorical_candidates:
    if cand != numeric_field_id and main_df[cand].nunique() < 10:
        group_field = cand
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()  # example aggregation
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field, and if possible, show grouping by the selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping field is available, visualize group differences
if group_field:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to explore the FAIR² clinical dataset, referencing all column and record set names by their `@id`. We loaded metadata and tabular records, performed a sample of exploratory data analysis—including filtering, normalization, and aggregation—and visualized distributions and group comparisons. This reproducible process, using a Croissant schema, can be extended to other FAIR datasets following the same referencing by `@id` and variable-driven workflow.